# Full-cohort feature extraction: band power

**Objective**: Extract band power features (delta, theta, alpha, beta, gamma; log-transformed per modelling_decisions.md Decision 2) across the full validated cohort, using the extract_subject_features orchestrator built and validated in 06_feature_extraction_pilot.ipynb. Assemble into a single feature matrix and save for the modelling phase.

**Inputs**: data/derivatives_heog_off/<subject_id>/sub-*_restEC-epo.fif, restEC condition, heog_off variant (primary per Decision 1).

**Assumptions**:

The subject list is derived by globbing for restEC epochs files directly, not from cohort_filtered_n163.xlsx or folder presence alone, so that the list matches exactly what can be loaded.
compute_band_power and extract_subject_features are treated as validated (pilot notebook, spot-checked to 15 decimal places) and are not re-derived here.
Subject-level extraction failures are logged, not fatal to the run.

In [11]:
# Imports and setup
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import mne
import glob
import pyarrow

# Temporary bootstrap path, just to make src/ importable — not the real project root
sys.path.insert(0, str(Path.cwd().parent))

from src.preprocessing import find_repo_root
project_root = find_repo_root()
data_dir = project_root / "data"

from src.features import extract_subject_features

print(f"Project root: {project_root}")
print(f"Data dir: {data_dir}")

Project root: /Users/romyweinstock/eeg-rtms-response-prediction
Data dir: /Users/romyweinstock/eeg-rtms-response-prediction/data


In [12]:
# Build the path to the derivatives_heog_off folder
derivatives_heog_off_path = data_dir / "derivatives_heog_off"

# Find every restEC epochs file, searching recursively since
sub_epoch_list = list(derivatives_heog_off_path.rglob("sub-*_restEC-epo.fif"))

# Check matches 
print(len(sub_epoch_list))
print(sub_epoch_list[0])

# Pull the subject ID out of each match
ID_list = []
for sub in sub_epoch_list:
    ID_list.append(sub.stem.split("_")[0])
assert len(ID_list) == 160, f"expected 160 rows got {len(ID_list)}"

160
/Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-88045809/sub-88045809_restEC-epo.fif


In [13]:
# Loop extract_subject_features across the full cohort
condition = "restEC"
variant = "heog_off"

qc_log = pd.read_csv(data_dir / 'batch_results_log_full_cohort.csv')

bands = {
    "delta": (2, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}

results_list = []
for subject_id in ID_list:
    result = extract_subject_features(subject_id, condition, variant, data_dir, qc_log, bands)
    results_list.append(result)

print(f"Processed {len(results_list)} subjects")

n_failed = sum(1 for r in results_list if r["reason"] != "ok")
print(f"Failures: {n_failed}")

Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-88045809/sub-88045809_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
21 matching events found
No baseline correction applied
0 projection items activated
Effective window size : 4.096 (s)
Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-88022765/sub-88022765_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Effective window size : 4.096 (s)
Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-88023485/sub-88023485_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensa

In [14]:
# Assemble into one dataframe
bandpower_full_cohort_df = pd.DataFrame(results_list)
print(bandpower_full_cohort_df.shape)
print(bandpower_full_cohort_df.columns.tolist())

(160, 146)
['subject_id', 'condition', 'variant', 'reason', 'heog_variant', 'preprocessing_status', 'n_epochs_before', 'n_epochs_after', 'output_path', 'autoreject_consensus', 'autoreject_n_interpolate', 'autoreject_extreme', 'heog_n_candidates', 'heog_n_valid', 'heog_correction_applied', 'preprocessing_error', 'Fp1_delta_power', 'Fp2_delta_power', 'F7_delta_power', 'F3_delta_power', 'Fz_delta_power', 'F4_delta_power', 'F8_delta_power', 'FC3_delta_power', 'FCz_delta_power', 'FC4_delta_power', 'T7_delta_power', 'C3_delta_power', 'Cz_delta_power', 'C4_delta_power', 'T8_delta_power', 'CP3_delta_power', 'CPz_delta_power', 'CP4_delta_power', 'P7_delta_power', 'P3_delta_power', 'Pz_delta_power', 'P4_delta_power', 'P8_delta_power', 'O1_delta_power', 'Oz_delta_power', 'O2_delta_power', 'Fp1_theta_power', 'Fp2_theta_power', 'F7_theta_power', 'F3_theta_power', 'Fz_theta_power', 'F4_theta_power', 'F8_theta_power', 'FC3_theta_power', 'FCz_theta_power', 'FC4_theta_power', 'T7_theta_power', 'C3_thet

In [15]:
# Missingness check across the full feature matrix
total_missing = bandpower_full_cohort_df.isna().sum().sum()
print(f"Total missing values: {total_missing}")

# If any missingness exists, break it down by column 
if total_missing > 0:
    missing_by_col = bandpower_full_cohort_df.isna().sum()
    print(missing_by_col[missing_by_col > 0])

Total missing values: 160
preprocessing_error    160
dtype: int64


In [16]:
# Confirm preprocessing_status is consistently non-null, supporting the
# interpretation that preprocessing_error is NaN because nothing failed
print(bandpower_full_cohort_df['preprocessing_status'].value_counts(dropna=False))

preprocessing_status
ok    160
Name: count, dtype: int64


**Missingness check**. preprocessing_error is NaN for all 160 subjects (160/160 missing)- expected, not a data quality issue. preprocessing_status confirms all 160 subjects preprocessed successfully ("ok"), and preprocessing_error is only ever populated on a preprocessing failure. No other columns show any missingness. Combined with reason == "ok" for all 160 rows (feature-extraction stage), both pipeline stages independently confirm a clean run.

In [17]:
# Per-subject check: does posterior alpha exceed frontal alpha for each subject
posterior_channels = ['O1', 'O2', 'Pz']
frontal_channels = ['Fp1', 'Fp2']

posterior_alpha = bandpower_full_cohort_df[[f"{ch}_alpha_power" for ch in posterior_channels]].mean(axis=1)
frontal_alpha = bandpower_full_cohort_df[[f"{ch}_alpha_power" for ch in frontal_channels]].mean(axis=1)

posterior_exceeds_frontal = posterior_alpha > frontal_alpha

n_subjects_matching = posterior_exceeds_frontal.sum()
pct_subjects_matching = posterior_exceeds_frontal.mean() * 100

print(f"{n_subjects_matching}/160 subjects show posterior > frontal alpha ({pct_subjects_matching:.1f}%)")

143/160 subjects show posterior > frontal alpha (89.4%)


In [18]:
# Do the non-matching subjects cluster with existing QC flags, or are they
# scattered independent of preprocessing quality?
non_matching = bandpower_full_cohort_df[~posterior_exceeds_frontal]

print(non_matching[['subject_id', 'n_epochs_after', 'autoreject_extreme', 'autoreject_consensus']])
print()
print("n_epochs_after — non-matching subjects:")
print(non_matching['n_epochs_after'].describe())
print("n_epochs_after — full cohort:")
print(bandpower_full_cohort_df['n_epochs_after'].describe())

       subject_id  n_epochs_after  autoreject_extreme  autoreject_consensus
10   sub-88059261            22.0               False                   0.2
30   sub-88059573            24.0                True                   1.0
31   sub-88066953            23.0               False                   0.2
34   sub-88012817            24.0                True                   1.0
39   sub-88020381            23.0               False                   0.1
45   sub-88012053            24.0                True                   1.0
47   sub-88061597            23.0               False                   0.2
48   sub-88043065            24.0                True                   1.0
50   sub-88025685            23.0               False                   0.2
53   sub-88023529            23.0               False                   0.2
69   sub-88052869            23.0               False                   0.2
75   sub-88044501            21.0               False                   0.5
86   sub-880

**Distribution check - posterior vs. frontal alpha, full cohort.** 143/160 subjects (89.4%) show posterior alpha (mean of O1, O2, Pz) exceeding frontal alpha (mean of Fp1, Fp2), consistent with the pilot subject and expected resting-EEG topography. The 17 non-matching subjects were checked against existing QC flags: their n_epochs_after (21-24, mean 23.1) falls within the full cohort's established range, and their autoreject_extreme rate (41%) is close to the cohort-wide baseline (~38%), showing no meaningful clustering on either measure. This suggests the non-matching subjects reflect genuine inter-subject variability rather than a QC-driven artifact, though this rules out only the two QC dimensions checked here, not every possible explanation.

In [19]:
# Save as Parquet file as it stores the dtype schema alongside the data
features_dir = data_dir / "features"
features_dir.mkdir(parents=True, exist_ok=True)

bandpower_full_cohort_df.to_parquet(features_dir / "bandpower_full_cohort.parquet")

In [20]:
reload_check = pd.read_parquet(features_dir / "bandpower_full_cohort.parquet")
print(reload_check.shape == bandpower_full_cohort_df.shape)
print(reload_check.equals(bandpower_full_cohort_df))

True
True


## Full-cohort band power extraction summary 

**Objective**: Extract band power features (5 bands × 26 channels) across the full validated cohort using the pilot-validated orchestrator (06), assemble into one feature matrix, QC it, and save.

**Result**: 160/160 subjects processed successfully (0 extraction failures). Assembled into a 160×146 dataframe (130 feature columns + 4 identity/status + 12 QC columns). Saved as data/features/bandpower_full_cohort.parquet, reload-verified for exact shape and value equality.

**QC findings**:

Missingness: only preprocessing_error is null (160/160), expected - populated only on preprocessing failure, and preprocessing_status confirms all 160 succeeded.
Distribution: 143/160 subjects (89.4%) show posterior alpha > frontal alpha, consistent with the pilot subject and expected resting-EEG topography. The 17 non-matching subjects show no meaningful clustering on n_epochs_after or autoreject_extreme relative to the cohort baseline, suggesting genuine inter-subject variability rather than a QC artifact (checked on these two dimensions only, not exhaustive).